# Thí nghiệm 2: Kiểm định giả thuyết (Hypothesis Testing)
## Mục tiêu: so sánh Enrollment giữa 2 nhóm trạng thái thử nghiệm

### Bối cảnh thực tế
- Nhóm A (Recruiting): thử nghiệm đang tuyển người
- Nhóm B (Completed): thử nghiệm đã hoàn tất
- Câu hỏi: **Enrollment trung bình giữa 2 nhóm có khác nhau thực sự không?**

### Giả thuyết kiểm định
- $H_0$: Không có khác biệt về trung bình Enrollment giữa 2 nhóm
- $H_1$: Có khác biệt về trung bình Enrollment giữa 2 nhóm

### So sánh 2 hướng tiếp cận
- **t-test (baseline, không resampling)**: cần giả định phân phối gần chuẩn và phương sai phù hợp
- **Permutation test (resampling)**: không cần giả định chuẩn, đáng tin hơn khi dữ liệu lệch hoặc có outlier

### Metric theo dõi
- p-value của t-test và permutation test
- Độ chênh lệch trung bình quan sát được (observed mean difference)

### Dataset: Chi tiết các cột

| Cột | Kiểu | Ý nghĩa | Ghi chú |
|---|---|---|---|
| `Rank` | int | Thứ tự bản ghi thử nghiệm | Chỉ để định danh, không dùng trực tiếp trong kiểm định |
| `NCT` | str | Mã định danh clinical trial | Ví dụ: NCTxxxxxx |
| `Title` | str | Tên nghiên cứu thử nghiệm | Mô tả nội dung thử nghiệm |
| **`Status`** | **str** | **Trạng thái thử nghiệm** | **⭐ Dùng để tách 2 nhóm: Recruiting vs Completed** |
| **`Enrollment`** | **int/float** | **Số người tham gia thử nghiệm** | **⭐ Biến số chính dùng để kiểm định giả thuyết** |

### Tại sao chọn dataset này?
- ✅ Có biến nhóm `Status` rõ ràng để xây dựng bài toán kiểm định 2 mẫu
- ✅ Biến `Enrollment` thường lệch và có outliers, phù hợp để thấy ưu điểm của permutation test
- ✅ Ý nghĩa thực tế cao trong bối cảnh clinical trials
- ✅ Dễ minh họa sự khác biệt giữa kiểm định tham số (t-test) và kiểm định resampling

In [16]:
# Bước 1: Chuẩn bị dataset cho Hypothesis Testing
# - Ưu tiên tải từ Google Drive bằng gdown
# - Nếu không tải được, fallback sang dataset mẫu để notebook luôn chạy được

import os, sys

if 'google.colab' in sys.modules:
    data_dir = '/content/data'
else:
    data_dir = os.path.abspath('../data') if os.path.exists('../data') else os.path.abspath('data')
os.makedirs(data_dir, exist_ok=True)

path = os.path.join(data_dir, 'covid_trials.csv')

if not os.path.exists(path):
    try:
        import importlib
        gdown = importlib.import_module('gdown')
        # Thay ID bên dưới nếu bạn có file dataset riêng trên Google Drive
        gdrive_id = '1jWOX6Vfqhu80l4uVvwdubHd0f0bsdgaJ' 
        url = f'https://drive.google.com/uc?id={gdrive_id}'
        print(f'⏳ Downloading from Google Drive: {url}')
        gdown.download(url, path, quiet=False)
        print(f'✅ Downloaded: {path}')
    except Exception as e:
        print(f'⚠️ Không tải được bằng gdown ({e}). Dùng dữ liệu mẫu fallback...')
        with open(path, 'w', encoding='utf-8') as f:
            f.write('Rank,NCT,Title,Status,Enrollment\n1,N1,A,Recruiting,100\n2,N2,B,Completed,520\n3,N3,C,Recruiting,150\n4,N4,D,Completed,180\n5,N5,E,Recruiting,90\n6,N6,F,Completed,610\n7,N7,G,Recruiting,80\n8,N8,H,Completed,130\n9,N9,I,Recruiting,500\n10,N10,J,Completed,115\n')
        print(f'✅ Fallback dataset created: {path}')
else:
    print(f'✅ File already exists: {path}')

✅ File already exists: /content/data/covid_trials.csv


In [17]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

def run_ht(g1, g2, n_p=2000):
    """
    Permutation test cho chênh lệch trung bình giữa 2 nhóm.

    Parameters:
    -----------
    g1, g2 : np.array
        Hai nhóm quan sát cần so sánh
    n_p : int
        Số lần hoán vị để tạo phân phối dưới H0

    Returns:
    --------
    tuple: (observed_diff, perm_distribution, p_value)
    """
    # Chênh lệch trung bình quan sát được
    obs = np.mean(g1) - np.mean(g2)

    # Gộp dữ liệu 2 nhóm để mô phỏng giả thuyết H0: không khác biệt
    c = np.concatenate([g1, g2])

    # Tạo phân phối hoán vị của thống kê kiểm định
    ps = np.array([np.mean(s[:len(g1)]) - np.mean(s[len(g1):]) for s in [np.random.permutation(c) for _ in range(n_p)]])

    # p-value hai phía
    return obs, ps, np.mean(np.abs(ps) >= np.abs(obs))

In [18]:
# Bước 2: Chạy thí nghiệm Hypothesis Testing
# - So sánh Enrollment giữa nhóm Recruiting và Completed
# - Dùng n=50 (giống thí nghiệm 1) để đồng nhất cách trình bày

import os
from scipy import stats

data_dir = '/content/data' if 'google.colab' in os.sys.modules else (os.path.abspath('../data') if os.path.exists('../data') else os.path.abspath('data'))
df = pd.read_csv(os.path.join(data_dir, 'covid_trials.csv'))

# Tách 2 nhóm gốc
g1_all = df[df['Status'] == 'Recruiting']['Enrollment'].dropna().values
g2_all = df[df['Status'] == 'Completed']['Enrollment'].dropna().values

# Lấy mẫu n=50 cho mỗi nhóm (replace=True nếu dữ liệu gốc nhỏ hơn 50)
n = 50
rng = np.random.default_rng(1)
g1 = rng.choice(g1_all, size=n, replace=len(g1_all) < n)
g2 = rng.choice(g2_all, size=n, replace=len(g2_all) < n)

# Baseline không resampling: Welch t-test
t_stat, p_t = stats.ttest_ind(g1, g2, equal_var=False)

# Resampling: Permutation test
obs, dist, p_p = run_ht(g1, g2)

print('📋 Group summary (n=50 mỗi nhóm):')
print(f"  - Recruiting: n={len(g1)}, mean={np.mean(g1):.2f}, std={np.std(g1):.2f}")
print(f"  - Completed: n={len(g2)}, mean={np.mean(g2):.2f}, std={np.std(g2):.2f}")
print(f"  - Observed mean difference (Recruiting - Completed): {obs:.2f}")
print(f"  - t-test p-value (baseline): {p_t:.4f}")
print(f"  - Permutation p-value (resampling): {p_p:.4f}")

sns.set_theme(style='whitegrid')
plt.figure(figsize=(10, 5))
sns.histplot(dist, kde=True, color='steelblue')
plt.axvline(obs, color='red', linestyle='--', linewidth=2, label=f'Observed diff = {obs:.2f}')
plt.title('Permutation Distribution of Mean Difference (n=50/group)')
plt.xlabel('Mean Difference under H0')
plt.ylabel('Frequency')
plt.legend()
plt.show()

KeyError: 'Status'

## 2.3 Nhận xét kết quả

- Nếu hai p-value đều nhỏ (ví dụ < 0.05): có bằng chứng bác bỏ $H_0$, tức là Enrollment trung bình khác biệt đáng kể giữa 2 nhóm.
- Nếu p-value t-test và permutation gần nhau: dữ liệu tương đối "đẹp", giả định của t-test không bị vi phạm mạnh.
- Nếu p-value lệch nhau rõ: nên ưu tiên permutation test vì ít phụ thuộc giả định phân phối.
- Ý nghĩa thực tế: với dữ liệu thử nghiệm lâm sàng có thể lệch và nhiễu, **resampling cho kết luận chắc hơn** so với chỉ dùng kiểm định tham số cổ điển.